# Lesson 2: Conditionals and loops

**Jumps change the order in which instructions run.** This lesson builds on [Lesson 1: Processor and memory](01_processor_and_memory.ipynb). We already know about registers, memory, and the instructions `LDI`, `MOV`, `ADD`, `SUB`, `LOAD`, `STORE`, `OUT`, and `HALT`. See the [README](README.md) for an overview of the machine.

We will work through the control-flow sections first: predict how programs run, check our predictions by stepping through instructions, and modify a loop. Sections **9–10** then connect an expression with its storage, syntax, and the assembler/simulator APIs. The three extensions at the end are **optional** notebook experiments; Lecture 2 also uses Extensions B and C for selected demonstrations. Each experiment starts with a new processor. Stepping cells within the same experiment build on one another. Running `step()` again executes the next instruction; to start over, rerun the cell containing `CPU(...)`. After restarting the kernel, run the setup cell first.


## 1. Where does PC point?

`PC` is the number of the instruction to execute next. Without a jump, it advances by one instruction. Labels and comments do not have instruction numbers. In the step table, follow the transition in **PC before → after**; the registers show their values **after** each step.

First, we display the initial state, execute one instruction, and let the program finish. Before running the cell, predict what changes after the first step and what the program eventually prints. `HALT` also counts as an executed instruction.


In [1]:
from pathlib import Path
import sys

candidates = (Path.cwd(), Path.cwd() / "mini8", Path.cwd().parent)
module_dir = next(
    (path.resolve() for path in candidates
     if (path / "mini8.py").is_file() and (path / "notebook_tools.py").is_file()),
    None,
)
if module_dir is None:
    raise FileNotFoundError("Open this notebook from the project folder or its mini8 folder.")
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

from notebook_tools import CPU
from mini8 import assemble, disassemble, Mini8Error

pc_cpu = CPU("""
LDI R0, 7
OUT R0
HALT
""", title="Sequential execution")
pc_cpu.show()
pc_cpu.step()
pc_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 7",0 → 1,7,0,0,0,=,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
1,Initial state,1,7,0,0,0,=,—,—
2,OUT R0,1 → 2,7,0,0,0,=,—,7
3,HALT,2 → 2 · HALT,7,0,0,0,=,—,—


## Lecture navigation

The [Lecture 2 slides](lectures/output/pdf/02_conditionals_and_languages.pdf) contain 28 slides. Use this map to move between the lecture and notebook; the [demonstration guide](lectures/demo_guide.md) provides instructor notes. Run the section **1** setup cell below after every kernel restart. Within a stepped experiment, run its cells in order.

| Slides | Notebook section | Demonstration |
|---|---|---|
| 2–6 | **1–4.2**, including **3.1** | D07: PC, jumps, unsigned comparison, and the stored flag |
| 8–10, 24 | **5** | D08: an `if` statement and its boundary case |
| 11–14, 24 | **6–6.2** | D09: initialization, one iteration, and termination |
| 15–16 | **7**, **Extension C** | D10: a modified loop and an execution limit |
| 17 | **Extension B** | D11: variable storage in registers and memory |
| 18, 20–23 | **9–9.1** | D12: a complete expression, a grammar fragment, and an AST |
| 19, 25–26 | **10–10.1** | D13: assembly errors and translation/execution APIs |
| 28 | **8** | D14: the final comparison-flag prediction |

Slides 1, 7, and 27 connect the demonstrations with the lecture's broader concepts. The grammar and AST are illustrations of **manual translation**, not an implemented higher-level compiler.


## 2. `JMP`: some instructions are skipped

`JMP finish` sets `PC` to the instruction marked by the label `finish:`. The jump is unconditional: it always happens. The processor does not execute the label itself; the assembler replaces it with an instruction number.

The program contains two output instructions. Predict whether it prints `10`, `20`, or both numbers. Then find the jump target in the instruction listing and check the execution trace to see which output instruction was skipped.


In [2]:
jump_cpu = CPU("""
LDI R0, 10
JMP finish
OUT R0
finish:
LDI R0, 20
OUT R0
HALT
""", title="Jump over an instruction")
jump_cpu.listing()
jump_cpu.run()


Instruction number (PC),Byte offset,Opcode,A,B,Decoded instruction
0 ← PC,0,01,00,0A,"LDI R0, 10"
1,3,08,03,00,JMP 3
2,6,0D,00,00,OUT R0
3,9,01,00,14,"LDI R0, 20"
4,12,0D,00,00,OUT R0
5,15,00,00,00,HALT


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 10",0 → 1,10,0,0,0,=,—,—
2,JMP 3,1 → 3 ↶ jump,10,0,0,0,=,—,—
3,"LDI R0, 20",3 → 4,20,0,0,0,=,—,—
4,OUT R0,4 → 5,20,0,0,0,=,—,20
5,HALT,5 → 5 · HALT,20,0,0,0,=,—,—


## 3. `CMP`: less than, equal to, greater than

`CMP R0, R1` compares two nonnegative values. It leaves the registers unchanged and stores the result in the comparison flag: `LT` means less than, `EQ` means equal to, and `GT` means greater than. The panel displays these states as `<`, `=`, and `>`. The flag is separate from registers `R0`–`R3`.

We will run three separate processors. Predict the flag when comparing `3` with `5`, `5` with `5`, and `8` with `5`. After `CMP`, also check that the register values have not changed.


In [3]:
for value in (3, 5, 8):
    compare_cpu = CPU(f"""
    LDI R0, {value}
    LDI R1, 5
    CMP R0, R1
    HALT
    """, title=f"Compare {value} with 5")
    compare_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 3",0 → 1,3,0,0,0,=,—,—
2,"LDI R1, 5",1 → 2,3,5,0,0,=,—,—
3,"CMP R0, R1",2 → 3,3,5,0,0,<,—,—
4,HALT,3 → 3 · HALT,3,5,0,0,<,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 5",0 → 1,5,0,0,0,=,—,—
2,"LDI R1, 5",1 → 2,5,5,0,0,=,—,—
3,"CMP R0, R1",2 → 3,5,5,0,0,=,—,—
4,HALT,3 → 3 · HALT,5,5,0,0,=,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 8",0 → 1,8,0,0,0,=,—,—
2,"LDI R1, 5",1 → 2,8,5,0,0,=,—,—
3,"CMP R0, R1",2 → 3,8,5,0,0,>,—,—
4,HALT,3 → 3 · HALT,8,5,0,0,>,—,—


### 3.1. D07 · Unsigned comparison: 255 is greater than 1

Lecture 1 showed that `0 − 1` wraps to the byte `11111111`, whose MINI-8 value is **255**. `CMP` uses that same **unsigned** interpretation: it compares values from 0 through 255.

Predict the flag after `CMP R0, R1` below. Does it become `LT` or `GT`? Check that the registers stay unchanged and that the program produces no output values: it contains no `OUT` instruction.


In [4]:
unsigned_cpu = CPU("""
LDI R0, 255
LDI R1, 1
CMP R0, R1
HALT
""", title="Unsigned comparison: 255 and 1")
unsigned_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 255",0 → 1,255,0,0,0,=,—,—
2,"LDI R1, 1",1 → 2,255,1,0,0,=,—,—
3,"CMP R0, R1",2 → 3,255,1,0,0,>,—,—
4,HALT,3 → 3 · HALT,255,1,0,0,>,—,—


## 4. The flag remembers the last comparison

**Only `CMP` changes the flag.** Neither `ADD` nor `SUB` recalculates it. A conditional jump uses the flag currently stored.

In this experiment, we will proceed one cell at a time. First, we load two values and execute the first `CMP`. Before running the cell, predict the registers and flag after the third instruction. Then pause and inspect the state.


In [5]:
flag_cpu = CPU("""
LDI R0, 2
LDI R1, 5
CMP R0, R1
ADD R0, R1
CMP R0, R1
HALT
""", title="Only CMP changes the comparison flag")
flag_cpu.step(3)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 2",0 → 1,2,0,0,0,=,—,—
2,"LDI R1, 5",1 → 2,2,5,0,0,=,—,—
3,"CMP R0, R1",2 → 3,2,5,0,0,<,—,—


### 4.1. Does addition change the flag too?

The next instruction is `ADD R0, R1`. Before running the cell, predict the new value of `R0` and the flag. Execute one step and compare the state with the previous table.


In [6]:
flag_cpu.step()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
3,Initial state,3,2,5,0,0,<,—,—
4,"ADD R0, R1",3 → 4,7,5,0,0,<,—,—


### 4.2. Compare the new values

The flag still describes the original comparison of `2` with `5`, even though `R0` now contains `7`. We will now execute a new `CMP R0, R1`.

Predict its result. The first Python call below performs the comparison and displays the state; the second finishes the program by executing `HALT`.


In [7]:
flag_cpu.step()
flag_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
4,Initial state,4,7,5,0,0,<,—,—
5,"CMP R0, R1",4 → 5,7,5,0,0,>,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
5,Initial state,5,7,5,0,0,>,—,—
6,HALT,5 → 5 · HALT,7,5,0,0,>,—,—


## 5. `if`: a conditional jump and a boundary case

Source-level example (unsigned bytes; arithmetic wraps modulo 256):

```text
if x < 10:
    y = 1
else:
    y = 2
print(y)
```

We want the rule “if `x < 10`, print `1`; otherwise, print `2`.” `JGE` jumps when the flag is `EQ` or `GT`, meaning greater than or equal to. We therefore translate the condition into **a jump to the `else` branch when `x >= 10`**. The unconditional `JMP` then prevents execution from continuing from the first branch into the second.

We will run the program with `x = 7` and the boundary value `x = 10`. Predict the output in both cases. Then use the tables to identify which jumps were taken. The value `10` does not belong in the first branch.

These Python-like examples describe a **teaching language with unsigned byte values**, not all of Python's semantics. Addition and subtraction wrap modulo 256, comparisons are unsigned, and variables must be initialized before they are read.


In [8]:
for x in (7, 10):
    if_cpu = CPU(f"""
    LDI R0, {x}      ; R0 = x
    LDI R1, 10      ; R1 = boundary
    CMP R0, R1
    JGE otherwise
    LDI R2, 1       ; R2 = y
    JMP print_result
otherwise:
    LDI R2, 2
print_result:
    OUT R2
    HALT
    """, title=f"If statement: x = {x}")
    if_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 7",0 → 1,7,0,0,0,=,—,—
2,"LDI R1, 10",1 → 2,7,10,0,0,=,—,—
3,"CMP R0, R1",2 → 3,7,10,0,0,<,—,—
4,JGE 6,3 → 4 · no jump,7,10,0,0,<,—,—
5,"LDI R2, 1",4 → 5,7,10,1,0,<,—,—
6,JMP 7,5 → 7 ↶ jump,7,10,1,0,<,—,—
7,OUT R2,7 → 8,7,10,1,0,<,—,1
8,HALT,8 → 8 · HALT,7,10,1,0,<,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 10",0 → 1,10,0,0,0,=,—,—
2,"LDI R1, 10",1 → 2,10,10,0,0,=,—,—
3,"CMP R0, R1",2 → 3,10,10,0,0,=,—,—
4,JGE 6,3 → 6 ↶ jump,10,10,0,0,=,—,—
5,"LDI R2, 2",6 → 7,10,10,2,0,=,—,—
6,OUT R2,7 → 8,10,10,2,0,=,—,2
7,HALT,8 → 8 · HALT,10,10,2,0,=,—,—


## 6. `while`: compare, exit if needed, repeat

Source-level example (unsigned bytes; arithmetic wraps modulo 256):

```text
total = 0
i = 1
while i < 6:
    total = total + i
    i = i + 1
print(total)
```

We will add the numbers `1` through `5`. Each register has a fixed role: **`R0 = total`, `R1 = i`, `R2 = bound`, `R3 = step`**. The upper bound, `6`, is excluded from the sum.

The loop consists of a comparison, a conditional jump out of the loop, the loop body, and a jump back. The condition is checked **before every iteration**. For now, we will execute only the four initialization instructions. Predict all four registers and the instruction number that `PC` will point to.

The next two cells continue on the same processor. Rerunning a stepping cell executes more instructions; to restart this experiment, rerun the following cell containing `CPU(...)`.


In [9]:
sum_cpu = CPU("""
LDI R0, 0       ; R0 = total
LDI R1, 1       ; R1 = i
LDI R2, 6       ; R2 = bound (exclusive)
LDI R3, 1       ; R3 = step
loop:
CMP R1, R2
JGE finish
ADD R0, R1
ADD R1, R3
JMP loop
finish:
OUT R0
HALT
""", title="Sum from 1 to 5")
sum_cpu.step(4)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 0",0 → 1,0,0,0,0,=,—,—
2,"LDI R1, 1",1 → 2,0,1,0,0,=,—,—
3,"LDI R2, 6",2 → 3,0,1,6,0,=,—,—
4,"LDI R3, 1",3 → 4,0,1,6,1,=,—,—


### 6.1. Complete one iteration

The five instructions from `CMP` through `JMP` check the condition, add a number, increase the counter, and return to the condition. Before running the cell, predict **`total`, `i`, and `PC` after these five steps**.

Then find the instruction that changes the total and the instruction that changes the counter. Also notice how `PC` returns to the start of the loop.


In [10]:
sum_cpu.step(5)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
4,Initial state,4,0,1,6,1,=,—,—
5,"CMP R1, R2",4 → 5,0,1,6,1,<,—,—
6,JGE 9,5 → 6 · no jump,0,1,6,1,<,—,—
7,"ADD R0, R1",6 → 7,1,1,6,1,<,—,—
8,"ADD R1, R3",7 → 8,1,2,6,1,<,—,—
9,JMP 4,8 → 4 ↶ jump,1,2,6,1,<,—,—


### 6.2. Finish the remaining iterations

The processor has already completed one iteration. Before running the cell, predict how many more times it will execute the body, what it will eventually print, and the final value of the counter `i`.

`run()` continues from the current position. At the final condition check, the body is skipped: `JGE` jumps directly to the output instruction and `HALT`. After the run, find this final check in the table.


In [11]:
sum_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
9,Initial state,4,1,2,6,1,<,—,—
10,"CMP R1, R2",4 → 5,1,2,6,1,<,—,—
11,JGE 9,5 → 6 · no jump,1,2,6,1,<,—,—
12,"ADD R0, R1",6 → 7,3,2,6,1,<,—,—
13,"ADD R1, R3",7 → 8,3,3,6,1,<,—,—
14,JMP 4,8 → 4 ↶ jump,3,3,6,1,<,—,—
15,"CMP R1, R2",4 → 5,3,3,6,1,<,—,—
16,JGE 9,5 → 6 · no jump,3,3,6,1,<,—,—
17,"ADD R0, R1",6 → 7,6,3,6,1,<,—,—
18,"ADD R1, R3",7 → 8,6,4,6,1,<,—,—


## 7. Your modification: add the even numbers

Below is another independent, working program that calculates `1 + 2 + 3 + 4 + 5`. **Modify it to calculate `2 + 4 + 6 + 8 + 10`.** Change only three initial values: the first number, the exclusive upper bound, and the step. Keep the initial total at zero.

Before running the cell, write down the expected sum and the number of times the loop body will execute. After running it, check the counter and output in the table. Hint: the last number added must still satisfy `i < bound`. Later sections of the notebook do not depend on your solution.


In [12]:
exercise_cpu = CPU("""
LDI R0, 0       ; Initial total
LDI R1, 1       ; First value
LDI R2, 6       ; Exclusive bound
LDI R3, 1       ; Step
loop:
CMP R1, R2
JGE finish
ADD R0, R1
ADD R1, R3
JMP loop
finish:
OUT R0
HALT
""", title="Your loop experiment")
exercise_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 0",0 → 1,0,0,0,0,=,—,—
2,"LDI R1, 1",1 → 2,0,1,0,0,=,—,—
3,"LDI R2, 6",2 → 3,0,1,6,0,=,—,—
4,"LDI R3, 1",3 → 4,0,1,6,1,=,—,—
5,"CMP R1, R2",4 → 5,0,1,6,1,<,—,—
6,JGE 9,5 → 6 · no jump,0,1,6,1,<,—,—
7,"ADD R0, R1",6 → 7,1,1,6,1,<,—,—
8,"ADD R1, R3",7 → 8,1,2,6,1,<,—,—
9,JMP 4,8 → 4 ↶ jump,1,2,6,1,<,—,—


<details>
<summary>Open after trying the exercise: the three initial values</summary>

Set the first number to **2**, the exclusive upper bound to **11**, and the step to **2**:

```asm
LDI R1, 2
LDI R2, 11
LDI R3, 2
```

The upper bound can also be **12**: the values `2, 4, 6, 8, 10` satisfy the condition, but the next value, `12`, does not. The body executes **five times**, the output is **30**, and the final counter `i` is **12**. The initial value of `R0` stays `0`.

</details>


## 8. Final check: does the jump use the new values?

Before running the cell, answer these questions: **Which flag does `CMP` set? Does `SUB` change it? Will `LDI R0, 99` execute? What will be printed?** Follow the stored flag, the instruction order, and the jump target.


In [13]:
review_cpu = CPU("""
LDI R0, 4
LDI R1, 4
CMP R0, R1
SUB R0, R1
JGE print_result
LDI R0, 99
print_result:
OUT R0
HALT
""", title="Predict the branch and output")
review_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 4",0 → 1,4,0,0,0,=,—,—
2,"LDI R1, 4",1 → 2,4,4,0,0,=,—,—
3,"CMP R0, R1",2 → 3,4,4,0,0,=,—,—
4,"SUB R0, R1",3 → 4,0,4,0,0,=,—,—
5,JGE 6,4 → 6 ↶ jump,0,4,0,0,=,—,—
6,OUT R0,6 → 7,0,4,0,0,=,—,0
7,HALT,7 → 7 · HALT,0,4,0,0,=,—,—


<details>
<summary>Show the explanation for the final check</summary>

`CMP` compares `4` with `4`, so it sets `EQ`. `SUB` changes `R0` to `0` but leaves the flag at `EQ`. With `EQ`, `JGE` jumps to `print_result`, skipping the instruction that loads `99`. `OUT` prints **0**. If we inserted a new `CMP R0, R1` before the jump, it would set `LT` and the jump would not be taken. The original program executes **7 instructions, including `HALT`**.

</details>

Next, we will connect a complete expression with its storage and examine the assembler and simulator interfaces directly.


## 9. D12 · A complete expression and its storage map

Consider the source-level idea:

```text
x = 3
y = 4
w = 2
z = (x + y) - w
print(z)
```

We **translate this example by hand**. The project provides an assembler and a processor simulator; it does not compile this higher-level source. We choose data addresses **x → 10, y → 11, w → 12, z → 13**. Each `.equ` names an address; the initialization instructions store the values.

The six instructions from `LOAD R0, [x]` through `STORE [z], R0` calculate and assign the expression. `R0` holds the partial result; `R1` supplies the next operand. Predict **z**, the output, and all four watched memory cells before running the cell. Arithmetic still wraps modulo 256, although this input needs no wraparound.


In [14]:
expression_cpu = CPU("""
.equ x, 10
.equ y, 11
.equ w, 12
.equ z, 13

LDI R0, 3
STORE [x], R0
LDI R0, 4
STORE [y], R0
LDI R0, 2
STORE [w], R0

LOAD R0, [x]
LOAD R1, [y]
ADD R0, R1
LOAD R1, [w]
SUB R0, R1
STORE [z], R0

LOAD R0, [z]
OUT R0
HALT
""", watch=(10, 11, 12, 13), title="An expression and its storage map")
expression_cpu.run()
expression_cpu.memory(start=10, stop=14)
print("Output:", expression_cpu.output)
print("Executed instructions:", expression_cpu.steps)


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 3",0 → 1,3,0,0,0,=,—,—
2,"STORE [10], R0",1 → 2,3,0,0,0,=,[10]: 0 → 3,—
3,"LDI R0, 4",2 → 3,4,0,0,0,=,—,—
4,"STORE [11], R0",3 → 4,4,0,0,0,=,[11]: 0 → 4,—
5,"LDI R0, 2",4 → 5,2,0,0,0,=,—,—
6,"STORE [12], R0",5 → 6,2,0,0,0,=,[12]: 0 → 2,—
7,"LOAD R0, [10]",6 → 7,3,0,0,0,=,—,—
8,"LOAD R1, [11]",7 → 8,3,4,0,0,=,—,—
9,"ADD R0, R1",8 → 9,7,4,0,0,=,—,—


Output: (5,)
Executed instructions: 15


### 9.1. D12 · From expression syntax to an abstract syntax tree

A grammar describes which source forms are allowed. This **conceptual grammar fragment** allows names, unsigned integer literals, parentheses, addition, subtraction, and assignment:

```text
assignment ::= NAME "=" expression
expression ::= atom { ("+" | "-") atom }
atom       ::= NAME | NUMBER | "(" expression ")"
```

Here, `NAME` and `NUMBER` stand for tokens. Literal values must be in 0–255, arithmetic wraps modulo 256, and a variable must be initialized before reading it. The assignment target may be a new variable. Braces mean zero or more repetitions. We choose to group repeated additions and subtractions **from left to right**. This fragment does not define an entire language or supply a parser.

A possible **abstract syntax tree (AST)** records the structure of `z = (x + y) - w`:

```text
Assign
├── target: Name(z)
└── value: Sub
    ├── Add
    │   ├── Name(x)
    │   └── Name(y)
    └── Name(w)
```

The tree groups `x + y` before the subtraction. Connect each part with the instructions above: names provide loads, `Add` becomes `ADD`, `Sub` becomes `SUB`, and assignment ends with `STORE [z], R0`. The storage map supplies the addresses.

This is a **manual translation illustration**. No cell parses the grammar, builds an AST, or implements a higher-level compiler. This translation uses two temporary registers; arbitrary nested expressions would require a broader storage strategy.

**Self-check:** draw the expression AST for `a - (b - c)`. Which subtraction supplies the right operand of the outer subtraction?

<details>
<summary>Reveal the grouping</summary>

`Sub(Name(a), Sub(Name(b), Name(c)))`. The subtree `b - c` supplies the outer subtraction's right operand. This differs from the default left grouping of `a - b - c`.

</details>

A correct translation must preserve the source program's specified behavior. For these examples, compare output, final source-variable values on terminating runs, and termination. Temporary registers and instruction counts may differ between implementations.


## 10. D13 · Assemble, inspect, step, and run

The notebook panel delegates instruction execution to the simulator. We can also use the underlying functions directly:

| Interface | What it does | Result |
|---|---|---|
| `assemble(source)` | Translates assembly text into encoded instructions | `bytes` |
| `disassemble(code)` | Reconstructs assembly with numeric operands | Text |
| `Machine(code)` | Creates a fresh processor | A stateful machine |
| `machine.step()` | Executes one instruction on that machine | Before/after snapshots |
| `run(code)` | Executes the program on a **fresh** machine | A final result |

Unlike `run(code)`, the notebook wrapper's **`cpu.run()` continues the existing processor**. Watch the receiver and argument. Neither assembly nor disassembly executes the program.

Predict the first register change, the output from the fresh run, and whether disassembling then reassembling preserves the bytes. Names and comments are not recovered by disassembly.


In [15]:
from mini8 import Machine, run

api_source = "LDI R0, 3\nLDI R1, 4\nADD R0, R1\nOUT R0\nHALT\n"
api_code = assemble(api_source)
print("Machine bytes:", api_code.hex(" "))
print("Program size:", len(api_code), "bytes")
print(disassemble(api_code))

api_machine = Machine(api_code)
api_first_step = api_machine.step()
print("Before one step:", api_first_step.before.registers)
print("After one step:", api_first_step.after.registers)

api_result = run(api_code)
print("Output from a fresh run:", api_result.output)
print("Fresh run steps:", api_result.steps)
print("Separately stepped machine steps:", api_machine.state().steps)
print("Binary round trip:", assemble(disassemble(api_code)) == api_code)


Machine bytes: 01 00 03 01 01 04 05 00 01 0d 00 00 00 00 00
Program size: 15 bytes
LDI R0, 3              ; PC=000  01 00 03
LDI R1, 4              ; PC=001  01 01 04
ADD R0, R1             ; PC=002  05 00 01
OUT R0                 ; PC=003  0D 00 00
HALT                   ; PC=004  00 00 00

Before one step: (0, 0, 0, 0)
After one step: (3, 0, 0, 0)
Output from a fresh run: (7,)
Fresh run steps: 5
Separately stepped machine steps: 1
Binary round trip: True


### 10.1. D13 · An assembly error before execution

`ADD` needs two register operands. The malformed instruction `ADD R0` supplies only one. Predict whether the assembler can produce a complete program.

The assembler rejects this source **before execution**. We catch the expected `Mini8Error` so that **Run All** still completes. This is different from the execution limit in Extension C, which occurs after instructions have run.

This completes the main notebook sequence. The remaining extensions provide further experiments with multiplication, storage choices, and termination.


In [16]:
try:
    assemble("ADD R0\nHALT")
except Mini8Error as error:
    print("Expected assembly error:", error)


Expected assembly error: Line 1: ADD requires 2 operands.


## Extension A: multiplication without `MUL` (optional)

Source-level example (unsigned bytes; arithmetic wraps modulo 256):

```text
result = 0
while count != 0:
    result = result + addend
    count = count - 1
print(result)
```

We can build multiplication from repeated addition. `R0` holds the result, `R1` holds the number of repetitions remaining, and `R2` holds the value to add. The new jump instruction, `JE`, is taken only when the flag is `EQ`.

Predict both `3 × 4` and `0 × 4`. When the repetition count is zero, the program must exit **before the first addition or subtraction**; otherwise, subtracting `1` from zero would produce `255`. The multiplication result is also limited to eight bits and is calculated modulo `256`.


In [17]:
for count in (3, 0):
    multiply_cpu = CPU(f"""
    LDI R0, 0       ; Result
    LDI R1, {count} ; Remaining repetitions
    LDI R2, 4       ; Addend
loop:
    LDI R3, 0
    CMP R1, R3
    JE finish
    ADD R0, R2
    LDI R3, 1
    SUB R1, R3
    JMP loop
finish:
    OUT R0
    HALT
    """, title=f"Repeated addition: {count} times 4")
    multiply_cpu.run()


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 0",0 → 1,0,0,0,0,=,—,—
2,"LDI R1, 3",1 → 2,0,3,0,0,=,—,—
3,"LDI R2, 4",2 → 3,0,3,4,0,=,—,—
4,"LDI R3, 0",3 → 4,0,3,4,0,=,—,—
5,"CMP R1, R3",4 → 5,0,3,4,0,>,—,—
6,JE 10,5 → 6 · no jump,0,3,4,0,>,—,—
7,"ADD R0, R2",6 → 7,4,3,4,0,>,—,—
8,"LDI R3, 1",7 → 8,4,3,4,1,>,—,—
9,"SUB R1, R3",8 → 9,4,2,4,1,>,—,—


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 0",0 → 1,0,0,0,0,=,—,—
2,"LDI R1, 0",1 → 2,0,0,0,0,=,—,—
3,"LDI R2, 4",2 → 3,0,0,4,0,=,—,—
4,"LDI R3, 0",3 → 4,0,0,4,0,=,—,—
5,"CMP R1, R3",4 → 5,0,0,4,0,=,—,—
6,JE 10,5 → 10 ↶ jump,0,0,4,0,=,—,—
7,OUT R0,10 → 11,0,0,4,0,=,—,0
8,HALT,11 → 11 · HALT,0,0,4,0,=,—,—


## Extension B: the same sum with different variable storage (optional)

The first program keeps the total and counter in registers. The second stores them in data memory: `index` at address `0` and `total` at address `1`. `.equ` names the addresses; the `STORE` instructions write the initial values.

Both programs print `15`, but the second needs repeated data transfers. Compare the number of instructions actually executed: **33 versus 66, including `HALT`**. This applies to these two specific programs and inputs; it is not a general ratio of processor speeds. Long traces display at most 60 executed instructions; this does not limit the actual number of steps.


In [18]:
register_cpu = CPU("""
LDI R0, 0
LDI R1, 1
LDI R2, 6
LDI R3, 1
loop:
CMP R1, R2
JGE finish
ADD R0, R1
ADD R1, R3
JMP loop
finish:
OUT R0
HALT
""", title="Variables in registers")

memory_cpu = CPU("""
.equ index, 0
.equ total, 1
LDI R0, 1
STORE [index], R0
LDI R0, 0
STORE [total], R0
loop:
LOAD R0, [index]
LDI R1, 6
CMP R0, R1
JGE finish
LOAD R1, [total]
ADD R1, R0
STORE [total], R1
LDI R1, 1
ADD R0, R1
STORE [index], R0
JMP loop
finish:
LOAD R0, [total]
OUT R0
HALT
""", watch=(0, 1), title="Variables in memory")

register_cpu.run()
memory_cpu.run()
memory_cpu.memory(start=0, stop=2)
print(f"Registers: output={register_cpu.output}, steps={register_cpu.steps}")
print(f"Memory:    output={memory_cpu.output}, steps={memory_cpu.steps}")


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R0, 0",0 → 1,0,0,0,0,=,—,—
2,"LDI R1, 1",1 → 2,0,1,0,0,=,—,—
3,"LDI R2, 6",2 → 3,0,1,6,0,=,—,—
4,"LDI R3, 1",3 → 4,0,1,6,1,=,—,—
5,"CMP R1, R2",4 → 5,0,1,6,1,<,—,—
6,JGE 9,5 → 6 · no jump,0,1,6,1,<,—,—
7,"ADD R0, R1",6 → 7,1,1,6,1,<,—,—
8,"ADD R1, R3",7 → 8,1,2,6,1,<,—,—
9,JMP 4,8 → 4 ↶ jump,1,2,6,1,<,—,—


Registers: output=(15,), steps=33
Memory:    output=(15,), steps=66


## Extension C: a loop that never ends (optional)

This program always returns to the same label. It has neither an exit condition nor a `HALT` instruction. Predict which values will change and which instructions will repeat.

We will stop the simulator after `12` executed instructions and catch the expected error with `try`/`except`, so the entire notebook can run successfully. The limit is a simulator safeguard, not part of the instruction set. When the limit is reached, the processor **has not executed `HALT`**; we have simply stopped executing further steps.

**A less obvious non-terminating loop:** use the earlier sum loop with first value `2`, step `2`, and bound `255`. The byte counter visits `2, 4, …, 254, 0, 2, …`. It never reaches 255, so the unsigned condition `i < 255` always holds. Increasing a finite-width counter does not guarantee termination. An unbounded-integer version instead reaches 256 and exits.


In [19]:
infinite_cpu = CPU("""
LDI R1, 1
loop:
ADD R0, R1
JMP loop
""", title="Intentional infinite loop")

try:
    infinite_cpu.run(max_steps=12)
except Mini8Error as error:
    print(f"Expected simulator stop: {error}")
    infinite_cpu.history(limit=12)
    infinite_cpu.show()

print(f"Executed steps: {infinite_cpu.steps}")
print(f"Halted: {infinite_cpu.halted}")


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R1, 1",0 → 1,0,1,0,0,=,—,—
2,"ADD R0, R1",1 → 2,1,1,0,0,=,—,—
3,JMP 1,2 → 1 ↶ jump,1,1,0,0,=,—,—
4,"ADD R0, R1",1 → 2,2,1,0,0,=,—,—
5,JMP 1,2 → 1 ↶ jump,2,1,0,0,=,—,—
6,"ADD R0, R1",1 → 2,3,1,0,0,=,—,—
7,JMP 1,2 → 1 ↶ jump,3,1,0,0,=,—,—
8,"ADD R0, R1",1 → 2,4,1,0,0,=,—,—
9,JMP 1,2 → 1 ↶ jump,4,1,0,0,=,—,—


Expected simulator stop: The program did not reach HALT within 12 additional instructions. It may contain an infinite loop.


Step,Executed instruction,PC before → after,R0,R1,R2,R3,CMP,Memory write,New output
0,Initial state,0,0,0,0,0,=,—,—
1,"LDI R1, 1",0 → 1,0,1,0,0,=,—,—
2,"ADD R0, R1",1 → 2,1,1,0,0,=,—,—
3,JMP 1,2 → 1 ↶ jump,1,1,0,0,=,—,—
4,"ADD R0, R1",1 → 2,2,1,0,0,=,—,—
5,JMP 1,2 → 1 ↶ jump,2,1,0,0,=,—,—
6,"ADD R0, R1",1 → 2,3,1,0,0,=,—,—
7,JMP 1,2 → 1 ↶ jump,3,1,0,0,=,—,—
8,"ADD R0, R1",1 → 2,4,1,0,0,=,—,—
9,JMP 1,2 → 1 ↶ jump,4,1,0,0,=,—,—


Executed steps: 12
Halted: False
